# Agentic RAG Demo — Retrieval as a Decision

This notebook demonstrates the agentic RAG graph from `agentic_rag.py` on three
queries that should take three different paths:

1. A greeting — no retrieval needed
2. A clear, well-matched question — one retrieval pass is enough
3. An ambiguous, narrow-terminology question — first retrieval is weak, the
   graph reformulates and retries before answering

To keep this notebook runnable without live API keys, the LLM and vector
store calls are mocked with small deterministic functions. In production,
these are swapped for real calls (see `agentic_rag.py` docstring — nodes
take the model/retriever as injected callables for exactly this reason).

In [1]:
import sys
sys.path.append("..")
from agentic_rag import build_agentic_rag_graph, initial_state

## 1. A tiny synthetic document corpus

In [2]:
CORPUS = [
    "Refund Policy: Damaged products are eligible for a full refund within "
    "30 days of delivery. Customers must submit photo evidence of the damage "
    "through the returns portal.",
    "Refund Policy: Non-damaged returns (change of mind) are accepted within "
    "14 days, subject to a 10% restocking fee.",
    "Vendor Risk Framework: Category-4 incidents (data exposure or system "
    "compromise) trigger the Tier-1 escalation path: notify the security lead "
    "within 1 hour, freeze vendor access, and open a formal incident report.",
    "Vendor Risk Framework: Category-1 to Category-3 incidents follow "
    "standard quarterly review; no immediate escalation is required.",
    "Onboarding Guide: New vendors complete a security questionnaire and a "
    "30-day probationary access period before full integration.",
]

## 2. Mock LLM and retriever functions

Deterministic stand-ins for router / retriever / confidence-scorer / reformulator / answer generators, so the graph can be tested without live API calls.

In [3]:
def mock_llm_decide(query: str) -> str:
    greetings = ["hi", "hello", "how are you", "hey", "thanks", "thank you"]
    if any(g in query.lower() for g in greetings):
        return "DIRECT"
    return "RETRIEVE"


def mock_vector_search(query: str, k: int):
    q_terms = set(query.lower().replace("?", "").split())
    scored = []
    for doc in CORPUS:
        d_terms = set(doc.lower().replace(":", "").replace(",", "").split())
        overlap = len(q_terms & d_terms)
        scored.append((overlap, doc))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [doc for score, doc in scored[:k] if score > 0] or [scored[0][1]]


def mock_llm_score(query: str, context: str) -> float:
    q_terms = set(query.lower().replace("?", "").split())
    c_terms = set(context.lower().replace(":", "").replace(",", "").split())
    if not q_terms:
        return 0.0
    overlap_ratio = len(q_terms & c_terms) / len(q_terms)
    # Purely overlap-driven scoring: vague/informal phrasing that shares few
    # terms with the corpus scores low, even if the right doc was retrieved,
    # which is a realistic failure mode -- confidence should reflect how well
    # the query maps onto the source language, not just whether *a* doc came
    # back.
    return min(1.0, overlap_ratio * 2)


def mock_llm_reformulate(query: str, original_query: str) -> str:
    # A canned rewrite mapping vague/informal phrasing to corpus terminology.
    # Triggers on the vendor-security query specifically; once already
    # rewritten to canonical terms, leaves it unchanged (idempotent).
    if "vendor" in query.lower() and "tier-1" not in query.lower():
        return "Tier-1 escalation path for category-4 vendor incident"
    return query


def mock_llm_answer_direct(query: str) -> str:
    return "Hi! I am doing well, thanks for asking. How can I help?"


def mock_llm_answer_with_context(query: str, context: str) -> str:
    return f"Based on the available policy: {context.splitlines()[0][:180]}..."

## 3. Build the graph

In [4]:
app = build_agentic_rag_graph(
    llm_decide=mock_llm_decide,
    vector_search=mock_vector_search,
    llm_score=mock_llm_score,
    llm_reformulate=mock_llm_reformulate,
    llm_answer_direct=mock_llm_answer_direct,
    llm_answer_with_context=mock_llm_answer_with_context,
)

## 4. Run three queries through three different paths

In [5]:
test_queries = [
    "Hi, how are you today?",
    "What is our standard refund window for damaged products?",
    "What do we do if a vendor has a really bad security problem?",
]

results = []
for q in test_queries:
    state = initial_state(q, max_retries=2)
    result = app.invoke(state)
    results.append(result)
    needs_retrieval = result["needs_retrieval"]
    retry_count = result["retry_count"]
    path_str = " -> ".join(result["path_taken"])
    answer = result["answer"]
    print(f"Query: {q}")
    print(f"  Retrieval used : {needs_retrieval}")
    print(f"  Retries        : {retry_count}")
    print(f"  Path taken     : {path_str}")
    print(f"  Answer         : {answer}")
    print()

Query: Hi, how are you today?
  Retrieval used : False
  Retries        : 0
  Path taken     : router -> direct_answer
  Answer         : Hi! I am doing well, thanks for asking. How can I help?

Query: What is our standard refund window for damaged products?
  Retrieval used : True
  Retries        : 0
  Path taken     : router -> retrieve -> confidence_check -> generate_answer
  Answer         : Based on the available policy: Refund Policy: Damaged products are eligible for a full refund within 30 days of delivery. Customers must submit photo evidence of the damage through the returns portal....

Query: What do we do if a vendor has a really bad security problem?
  Retrieval used : True
  Retries        : 1
  Path taken     : router -> retrieve -> confidence_check -> reformulate -> retrieve -> confidence_check -> generate_answer
  Answer         : Based on the available policy: Vendor Risk Framework: Category-4 incidents (data exposure or system compromise) trigger the Tier-1 escalati

## 5. Observed behavior

| Query | Path taken | Why |
|---|---|---|
| "Hi, how are you today?" | router -> direct_answer | Greeting, no retrieval needed |
| "Refund window for damaged products?" | router -> retrieve -> confidence_check -> generate_answer | Clear terminology match, one pass is enough |
| "What do we do if a vendor has a really bad security problem?" | router -> retrieve -> confidence_check -> reformulate -> retrieve -> confidence_check -> generate_answer | Informal phrasing shares few terms with the corpus, so confidence scores low even though the right doc was found; reformulation maps it to corpus wording ("Tier-1 escalation path for category-4 vendor incident"), second pass scores high and answers |

The key thing being tested here isn't just "did we get an answer" — it's
**which path the graph took to get there**. A test suite for an agentic
pipeline needs to assert on the path, not just the final text, or it will
silently pass even when the routing logic is broken.